In [1]:
import json

SPECIAL = {"<pad>", "<s>", "</s>", "<unk>", "", "|"}

def load_ipa_to_waxholm(ipa_path, wax_path):
    ipa = json.load(open(ipa_path, encoding="utf-8"))
    wax = json.load(open(wax_path, encoding="utf-8"))
    id2wax = {v: k for k, v in wax.items()}          # bridge through shared IDs
    return {sym: id2wax[i] for sym, i in ipa.items()}

def ipa_to_waxholm(text, mapping, sil="sil"):
    vocab = sorted((s for s in mapping if s not in SPECIAL), key=len, reverse=True)
    out = [sil]
    for word in text.replace("|", " ").split():
        if word in SPECIAL:
            continue
        i = 0
        while i < len(word):
            for sym in vocab:                        # longest match wins
                if word.startswith(sym, i):
                    out.append(mapping[sym])
                    i += len(sym)
                    break
            else:
                raise ValueError(f"unknown IPA {word[i:]!r} in {word!r}")
        out.append(sil)
    return out